## Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

## Đọc file

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DS108/Final_Project/Data/Gold_data.csv')

## Chia tập dữ liệu

In [ ]:
target = 'Price'
X = df.drop(target, axis=1)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Shape of train:', X_train.shape)
print('Shape of test:', X_test.shape)

Shape of train: (193792, 153)
Shape of test: (48448, 153)


In [ ]:
# Scale data
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
# Hàm tính R2 Adjust
def r2_adjusted(r2, n, p):
  r2_adjusted = 1 - (1 - r2) * (n - 1) / (n - p - 1)
  return r2_adjusted

## Ridge Regression

### Mô hình với tham số mặc định và đánh giá bằng cross-validation

In [ ]:
ridge_model = Ridge()

_mae_train = -cross_val_score(ridge_model, X_train, y_train, cv=10, scoring='neg_mean_absolute_error')
_r2_train = cross_val_score(ridge_model, X_train, y_train, cv=10, scoring='r2')

### Kết quả đánh giá bằng cross-validation

In [ ]:
print('Result on train data:\n')
print('   Mean MAE:', _mae_train.mean())
print('   MAE standard deviation:', _mae_train.std())
print('\n   Mean R2:', _r2_train.mean())
print('   R2 standard deviation:', _r2_train.std())
print('\n   Adjusted R2:', r2_adjusted(_r2_train.mean(), len(X_train), X_train.shape[1]))

Result on train data:

   Mean MAE: 539337.5095316826
   MAE standard deviation: 3866.656898108318

   Mean R2: 0.5491277152621216
   R2 standard deviation: 0.007010488696587411

   Adjusted R2: 0.5487714656645999


### Thực hiện dự đoán trên tập test

In [ ]:
ridge_model.fit(X_train, y_train)
y_pred = ridge_model.predict(X_test)

_mae_test = mean_absolute_error(y_test, y_pred)
_r2_test = r2_score(y_test, y_pred)

### Kết quả dự đoán trên tập test

In [ ]:
print('Result on test data:')
print('   Mean Absolute Error:', _mae_test)
print('   R2:', _r2_test)
print('   Adjusted R2:', r2_adjusted(_r2_test, len(X_test), X_test.shape[1]))

Result on test data:
   Mean Absolute Error: 537779.3546020027
   R2: 0.5546653646824553
   Adjusted R2: 0.553254502065907


### Tìm bộ siêu tham số tối ưu bằng gridSearch

In [ ]:
param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100],
              'fit_intercept': [True, False],
              'solver': ['auto', 'svd', 'cholesky', 'lsqr'],
              }
_grid_search = GridSearchCV(estimator=Ridge(), param_grid=param_grid, cv=10,
                           scoring='neg_mean_absolute_error', n_jobs=-1)

_grid_search.fit(X_train, y_train)
_best_ridge_model = _grid_search.best_estimator_
_best_alpha = _grid_search.best_params_['alpha']
_best_mae_train = -_grid_search.best_score_

print('Best parameter & score:')
print('   MAE: ', _best_mae_train)
print('   Parametes: ', _grid_search.best_params_)

Best parameter & score:
   MAE:  539329.8717428495
   Parametes:  {'alpha': 0.1, 'fit_intercept': True, 'solver': 'svd'}


### Huấn luyện và đánh giá bằng bộ siêu tham số tối ưu tìm được

In [ ]:
y_pred_tuned = _best_ridge_model.predict(X_test)
_mae_test_tuned = mean_absolute_error(y_test, y_pred_tuned)
_r2_test_tuned = r2_score(y_test, y_pred_tuned)

In [ ]:
print('Result on test data after tuned:')
print('   MAE:', _mae_test_tuned)
print('   R2:', _r2_test_tuned)
print('   Adjusted R2:', r2_adjusted(_r2_test_tuned, len(X_test), X_test.shape[1]))

Result on test data after tuned:
   MAE: 537772.3944505282
   R2: 0.5546778701830712
   Adjusted R2: 0.5532670471851422
